# Introduction

In this Kaggle notebook, we focus on enhancing the [MMLU dataset](https://www.kaggle.com/datasets/peiyuanliu2001/mmlu-dataset?select=test.csv), which contains over 100,000 rows, each featuring a question with four multiple-choice options (A, B, C, D). 

In this notebook, we will:

* **Generate Option "E":** We will generate a fifth multiple-choice option using gpt-3.5-turbo, for each question in the dataset. The aim is to make this option plausible yet incorrect.

* **Shuffle Options:** After generating option "E", we will shuffle the multiple-choice options for each question. This is a critical step to ensure that any model trained on this enhanced dataset does not inadvertently learn to identify option "E" as always being incorrect.


Check out the full dataset here

# Installations

In [ ]:
! pip install llm

# Load libraries

In [ ]:
import os
import llm
import random

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

In [ ]:
# Initial Dataset Location 
DATA_LOC = "/kaggle/input/mmlu-dataset/"

# Change to False to run entire dataset
TRIAL_RUN = True

# OpenAI API setup

In [ ]:
model = llm.get_model("gpt-3.5-turbo")

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['OPENAI_API_KEY'] = secrets.get_secret("openai")

model.key = os.environ['OPENAI_API_KEY']

# Read data

In [ ]:
train = []

for file in os.listdir(DATA_LOC):
    if '.csv' in file:
        path = DATA_LOC + file
        train.append(pd.read_csv(path))  

train = pd.concat(train, axis=0)
train.drop(['Unnamed: 0'], axis=1, inplace=True)
train = train.reset_index(drop=True)

if TRIAL_RUN: train = train.head()
    
train.shape

In [ ]:
train.head()

# Generate Option E

In [ ]:
def generate_prompt(row):
    return f'''
            
            Given the following multiple choice question, please generate an option "E" in the same format as the other options, which would be plausible but ultimately the wrong answer
            
            Question: {row.prompt}
                
            A: {row.A}
            B: {row.B}
            C: {row.C}
            D: {row.D}

            Answer: {row.answer}
            
            ''' 

In [ ]:
# Example prompt

print(generate_prompt(train.loc[0]))

In [ ]:
E = []

for row in tqdm(train.itertuples(), total = len(train)):
    
    response = model.prompt(generate_prompt(row))
    optE = response.text()
    optE = optE.split(":")[1].strip()
    
    E.append(optE)
    
# Option E
train['E'] = E

# Reorder columns
train = train[[col for col in train.columns if 'answer' not in col] + ['answer']]

train.head()

# Shuffle options

In [ ]:
# Function to shuffle columns A, B, C, D, E and update answer accordingly
def shuffle_row(row):
    options = ['A', 'B', 'C', 'D', 'E']
    shuffle_options = ['A', 'B', 'C', 'D', 'E']
    old_answer = row['answer']
    old_answer_value = row[old_answer]
    
    random.shuffle(shuffle_options)
    
    new_row = {
        'prompt': row['prompt'],
        'answer': options[shuffle_options.index(old_answer)],  # Update the answer key
    }
    
    for opt1, opt2 in zip(['A', 'B', 'C', 'D', 'E'], shuffle_options):
        new_row[opt1] = row[opt2]
    
    return pd.Series(new_row)


In [ ]:
# Shuffle each row
train_shuffled = train.apply(shuffle_row, axis=1)

# Reorder columns
train_shuffled = train_shuffled[train.columns]

train_shuffled.shape

In [ ]:
train.loc[0]

In [ ]:
train_shuffled.loc[0]

# Save dataset

In [ ]:
train_shuffled.to_csv("train.csv", index=False)